# Policy Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn

from battery_sim.battery.battery import Battery
from battery_sim.agents.policies.rule_based import ThresholdPolicy
from battery_sim.agents.policies.probabilistic import ProbabilisticThresholdPolicy
from battery_sim.agents.policies.continuous_probabilistic import ContinuousProbabilisticPolicy
from battery_sim.agents.policies.reinforce import REINFORCEPolicy
from battery_sim.agents.policies.dqn import DQNPolicy
from battery_sim.agents.policies.mppi import MPPIPolicy
from battery_sim.agents.policies.forecast_lp import ForecastLPPolicy
from battery_sim.agents.policies.oracle import OraclePolicy
from battery_sim.agents.policies._utils import make_mlp
from battery_sim.models.median_reversion import MedianReversionModel
from battery_sim.optimization.policy_optimizer import evaluate_full
from battery_sim.utils.plotting import plot_eval, plot_daily_revenue, compare_policies

In [2]:
# Load data
train_df = pd.read_csv('../data/NEM_NSW1_2025-01-01_2025-12-31_1h.csv')
test_df = pd.read_csv('../data/NEM_NSW1_2026-01-01_2026-03-01_1h.csv')

print(f'Train: {len(train_df)} rows ({train_df.iloc[0, 0]} to {train_df.iloc[-1, 0]})')
print(f'Test:  {len(test_df)} rows ({test_df.iloc[0, 0]} to {test_df.iloc[-1, 0]})')

Train: 8736 rows (2025-01-01 10:00:00 to 2025-12-31 09:00:00)
Test:  1416 rows (2026-01-01 10:00:00 to 2026-03-01 09:00:00)


In [ ]:
# Battery config
# capacity is 200.0 mwh, charge rate is 100mw per hour
# very roughly based on New England 1 in NSW: https://gateway.icn.org.au/projects/14788/pg-14788
battery = Battery(
    capacity_mwh=200.0,
    max_charge_rate_mw=100,
    max_discharge_rate_mw=100,
    efficiency=0.9,
)

In [4]:
Q = 20
vals = np.quantile(train_df.price, q = [i/Q for i in range(1, Q)])
PARAM_GRID = {
    'buy_threshold':vals,
    'sell_threshold':vals,
}

## 1. Deterministic Threshold Policy

In [ ]:
det_policy = ThresholdPolicy(
    buy_threshold=50.0,
    sell_threshold=200.0,
)

# Learn optimal thresholds on training data
det_policy.learn(
    train_data=train_df, 
    battery=battery, 
    num_iters=25, 
    window_len=24 * 7,
    param_grid=PARAM_GRID
)
print(f'Learned buy_threshold: {det_policy.buy_threshold}')
print(f'Learned sell_threshold: {det_policy.sell_threshold}')

## 2. Probabilistic Threshold Policy

In [ ]:
price_model = MedianReversionModel(window_days=30, interval_minutes=60)

prob_policy = ProbabilisticThresholdPolicy(
    model=price_model,
    buy_threshold=50.0,
    sell_threshold=200.0,
)

# Learn: fits model + grid searches thresholds
prob_policy.learn(
    train_data=train_df, 
    battery=battery, 
    num_iters=10, 
    window_len=24 * 7,
    param_grid=PARAM_GRID
)
print(f'Learned buy_threshold: {prob_policy.buy_threshold}')
print(f'Learned sell_threshold: {prob_policy.sell_threshold}')

## 3. Continuous Probabilistic Policy

In [ ]:
cont_model = MedianReversionModel(window_days=30, interval_minutes=60)

cont_policy = ContinuousProbabilisticPolicy(
    model=cont_model,
    buy_threshold=50.0,
    sell_threshold=200.0,
)

cont_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=10,
    window_len=24 * 7,
    param_grid=PARAM_GRID,
)
print(f'Learned buy_threshold: {cont_policy.buy_threshold}')
print(f'Learned sell_threshold: {cont_policy.sell_threshold}')

## 4. REINFORCE Policy (Policy Gradient)

In [ ]:
# REINFORCE: policy gradient with Beta action distribution (actions in [-1, 1])
reinforce_net = make_mlp(in_features=2, out_features=2, hidden=16)
reinforce_policy = REINFORCEPolicy(
    net=reinforce_net,
    lr=1e-3,
    gamma=0.99,
    entropy_coef=0.01,
)

reinforce_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=20,
    window_len=24 * 7,
    n_windows=25,
)

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax1.plot(reinforce_policy.loss_history)
ax1.set(xlabel='Episode', ylabel='Loss', title='REINFORCE Policy Loss')
ax2.plot(reinforce_policy.return_history)
ax2.set(xlabel='Episode', ylabel='Return', title='REINFORCE Episode Return')
plt.tight_layout()
plt.show()

## 5. DQN Policy (Deep Q-Learning)

In [ ]:
# DQN: discrete action Q-learning with replay buffer (actions in [-1, 1])
dqn_net = make_mlp(in_features=2, out_features=11, hidden=32)
dqn_policy = DQNPolicy(
    net=dqn_net,
    n_actions=11,
    lr=1e-3,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=500,
    batch_size=32,
    buffer_size=2000,
)

dqn_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=20,
    window_len=24 * 7,
    n_windows=25,
)

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax1.plot(dqn_policy.loss_history)
ax1.set(xlabel='Gradient Step', ylabel='TD Loss', title='DQN TD Loss')
ax2.plot(dqn_policy.epsilon_history)
ax2.set(xlabel='Episode', ylabel='Epsilon', title='DQN Epsilon Decay')
plt.tight_layout()
plt.show()

## 6. MPPI (Model Predictive Path Integral) Policy

In [ ]:
mppi_model = MedianReversionModel(window_days=30, interval_minutes=60)
mppi_policy = MPPIPolicy(
    model=mppi_model,
    battery=battery,
    horizon=12,
    n_samples=100,
    n_trajectories=50,
)

mppi_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=25,
    window_len=24 * 7,
)
print(f'Tuned noise_sigma: {mppi_policy.noise_sigma}')
print(f'Tuned temperature: {mppi_policy.temperature}')

## 7. Forecast LP Policy (MPC with Model Forecasts)

In [ ]:
forecast_lp_model = MedianReversionModel(window_days=30, interval_minutes=60)
forecast_lp_policy = ForecastLPPolicy(
    model=forecast_lp_model,
    battery=battery,
    horizon=12,
    interval_minutes=60,
)

forecast_lp_policy.learn(
    train_data=train_df,
    battery=battery,
)
print("Forecast LP policy trained")

## 8. Oracle (Perfect Foresight)

In [ ]:
oracle = OraclePolicy(price_data=test_df, battery=battery)
print(f'Oracle solved LP with {len(test_df) - 1} tradeable steps')

## 9. Evaluate on Test Set

In [ ]:
# Re-fit models on train data before test evaluation
mppi_model.fit(train_df)
cont_model.fit(train_df)
forecast_lp_model.fit(train_df)
price_model.fit(train_df)

det_eval = evaluate_full('Deterministic Threshold', det_policy, test_df, battery)
prob_eval = evaluate_full('Probabilistic Threshold', prob_policy, test_df, battery)
cont_eval = evaluate_full('Continuous Probabilistic', cont_policy, test_df, battery)
reinforce_eval = evaluate_full('REINFORCE', reinforce_policy, test_df, battery)
dqn_eval = evaluate_full('DQN', dqn_policy, test_df, battery)
mppi_eval = evaluate_full('MPPI', mppi_policy, test_df, battery)
forecast_lp_eval = evaluate_full('Forecast LP', forecast_lp_policy, test_df, battery)
oracle_eval = evaluate_full('Oracle', oracle, test_df, battery)

all_evals = [det_eval, prob_eval, cont_eval, reinforce_eval, dqn_eval, mppi_eval, forecast_lp_eval, oracle_eval]
for e in all_evals:
    print(e.summary())
    print()

## 10. Detailed Plots

In [ ]:
for e in all_evals:
    plot_eval(e)
    plt.show()

In [ ]:
for e in all_evals:
    plot_daily_revenue(e)
    plt.show()

## 11. Policy Comparison

In [ ]:
compare_policies(all_evals)
plt.show()